# Эксперимент: MFCC + SVM (baseline)

**Модель:** SVM (RBF kernel) на статистиках MFCC (mean + std + delta_mean + delta_std)  
**Группа:** 01_baselines

## Исправления по сравнению с checkpoint_3/exp_01_mfcc_svm

1. **Holdout test изолирован** до начала CV — тест не участвует в подборе гиперпараметров.
2. **5-fold StratifiedKFold** вместо одного train/val сплита → mean ± std по F1.
3. **Оптимизация порога** на каждом val-фолде по F1-bad вместо фиксированного 0.5.
4. **MLflow** — все параметры, метрики и артефакты логируются автоматически.

In [5]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import mlflow
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils, train_utils
from shared.evaluate import find_optimal_threshold, evaluate, evaluate_cv_folds, print_cv_summary
from shared.data_utils import build_feature_matrix, get_cv_folds
from shared.results_utils import save_result_csv
from shared.mlflow_utils import start_run, log_cv_fold

## 1. Holdout split

In [6]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_holdout_split()

print(f"Train+Val: {len(paths_trainval)}  |  Holdout Test: {len(paths_test)}")
print(f"Баланс test: good={np.sum(labels_test==0)}, bad={np.sum(labels_test==1)}")

Train+Val: 2359  |  Holdout Test: 417
Баланс test: good=282, bad=135


## 2. 5-Fold CV + обучение финальной модели

In [7]:
extractor = data_utils.extract_mfcc_stats

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "clf__gamma": ["scale", "auto", 0.01, 0.1],
}

with start_run("exp_mfcc_svm", group="01_baselines"):

    # --- Логируем гиперпараметры ---
    mlflow.log_params({
        "model":        "SVM RBF",
        "features":     "MFCC stats (mean+std+delta)",
        "n_mfcc":       config.N_MFCC,
        "cv_folds":     config.CV_N_SPLITS,
        "use_letters":  True,
        "class_weight": "balanced",
    })

    # --- 5-fold CV ---
    fold_results = []
    t0 = time.perf_counter()

    for paths_tr, paths_val, labels_tr, labels_val, letters_tr, letters_val, fold in \
            get_cv_folds(paths_trainval, labels_trainval, letters_trainval):

        print(f"\nФолд {fold+1}/{config.CV_N_SPLITS}")
        X_tr  = build_feature_matrix(paths_tr,  extractor)
        X_val = build_feature_matrix(paths_val, extractor)
        X_tr  = np.hstack([X_tr,  letters_tr])
        X_val = np.hstack([X_val, letters_val])

        grid = GridSearchCV(pipeline, param_grid, cv=3,
                            scoring="f1_macro", refit=True, n_jobs=-1)
        grid.fit(X_tr, labels_tr)
        clf = grid.best_estimator_
        print(f"  best_params={grid.best_params_}")

        y_proba_val = clf.predict_proba(X_val)[:, config.CLASS_BAD]
        thr = find_optimal_threshold(labels_val, y_proba_val)
        metrics = evaluate(labels_val, y_proba_val, threshold=thr, verbose=True)
        fold_results.append(metrics)

        # Логируем метрики фолда в MLflow
        log_cv_fold(fold, f1_bad=metrics["f1_bad"], f1_macro=metrics["f1_macro"],
                    roc_auc=metrics["roc_auc"], threshold=thr)

    train_time_sec = time.perf_counter() - t0
    cv_agg = evaluate_cv_folds(fold_results)
    print_cv_summary(cv_agg)

    # --- Финальная модель на train+val → holdout test ---
    X_trainval = build_feature_matrix(paths_trainval, extractor)
    X_test     = build_feature_matrix(paths_test,     extractor)
    X_trainval = np.hstack([X_trainval, letters_trainval])
    X_test     = np.hstack([X_test,     letters_test])

    final_grid = GridSearchCV(pipeline, param_grid, cv=5,
                              scoring="f1_macro", refit=True, n_jobs=-1)
    final_grid.fit(X_trainval, labels_trainval)
    best_clf = final_grid.best_estimator_
    print(f"\nЛучшие параметры (финал): {final_grid.best_params_}")

    # Логируем лучшие параметры
    mlflow.log_params({f"best_{k}": v for k, v in final_grid.best_params_.items()})

    cv_threshold = cv_agg["threshold_mean"]
    y_proba_test = best_clf.predict_proba(X_test)[:, config.CLASS_BAD]
    test_metrics = evaluate(labels_test, y_proba_test, threshold=cv_threshold, verbose=True)

    # --- Сохранение: CSV + автологирование в MLflow ---
    save_result_csv(
        exp_dir=exp_dir,
        experiment_id="exp_mfcc_svm",
        experiment_name="MFCC + SVM (baseline)",
        model="SVM RBF на MFCC stats",
        accuracy=test_metrics["accuracy"],
        f1_macro=test_metrics["f1_macro"],
        f1_bad=test_metrics["f1_bad"],
        roc_auc=test_metrics["roc_auc"],
        precision_bad=test_metrics["precision_bad"],
        recall_bad=test_metrics["recall_bad"],
        threshold=test_metrics["threshold"],
        cv_f1_bad_std=cv_agg["f1_bad_std"],
        cv_f1_macro_std=cv_agg["f1_macro_std"],
        notes=f"5-fold CV + holdout | best_params={final_grid.best_params_} | cv_thr={cv_threshold:.2f}",
        train_time_sec=train_time_sec,
    )
    print("\nРезультаты сохранены (result.csv + MLflow)")

/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/13 08:57:31 INFO mlflow.tracking.fluent: Experiment with name '01_baselines' does not exist. Creating a new experiment.



Фолд 1/5


/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/resampy/filters.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/resampy/filters.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/resampy/filters.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin 

  best_params={'clf__C': 1.0, 'clf__gamma': 'scale'}
              precision    recall  f1-score   support

        good       0.85      0.64      0.73       318
         bad       0.51      0.77      0.62       154

    accuracy                           0.69       472
   macro avg       0.68      0.71      0.68       472
weighted avg       0.74      0.69      0.70       472

Threshold : 0.30
Accuracy  : 0.6864
F1-macro  : 0.6757
F1-bad    : 0.6166
ROC-AUC   : 0.7697

Фолд 2/5
  best_params={'clf__C': 2.0, 'clf__gamma': 0.01}
              precision    recall  f1-score   support

        good       0.86      0.60      0.71       319
         bad       0.49      0.80      0.61       153

    accuracy                           0.67       472
   macro avg       0.68      0.70      0.66       472
weighted avg       0.74      0.67      0.68       472

Threshold : 0.28
Accuracy  : 0.6653
F1-macro  : 0.6582
F1-bad    : 0.6089
ROC-AUC   : 0.7522

Фолд 3/5
  best_params={'clf__C': 2.0, 'clf__g

## 3. Таблица по фолдам

In [8]:
df_folds = pd.DataFrame(fold_results)
df_folds.index = [f"Fold {i+1}" for i in range(len(fold_results))]
display(df_folds[["accuracy", "f1_macro", "f1_bad", "roc_auc", "threshold"]])

,accuracy,f1_macro,f1_bad,roc_auc,threshold
Fold 1,0.686441,0.675674,0.616580,0.769685,0.30
Fold 2,0.665254,0.658159,0.608911,0.752167,0.28
Fold 3,0.692797,0.681992,0.623377,0.774602,0.27
Fold 4,0.747881,0.726137,0.648968,0.783310,0.36
Fold 5,0.760085,0.736834,0.658610,0.807899,0.37
